In [ ]:
import pandas as pd
import csv
import random
import ipaddress
from datetime import datetime, timedelta

INPUT_FILE = "dataset_agrupado.csv"
OUTPUT_FILE = "dataset_expandido.csv"

NUM_USUARIOS_POOL = 100000

TEXT_TEMPLATES = {
    "Por la negativa a recibir el reclamo, recurso o queja; o por la negativa a otorgar el código de reclamo o queja": [
        "No quisieron aceptar mi reclamo.",
        "La empresa rechazó registrar mi queja.",
        "No me brindaron código de reclamo.",
        "Se negaron a procesar mi solicitud."
    ]
}

# =========================
# LECTURA ROBUSTA CSV
# =========================
rows = []

with open(INPUT_FILE, encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)

    for row in reader:
        if len(row) > 7:
            # Recombinar columnas extras dentro de Tipo de Queja
            row = row[:5] + [",".join(row[5:-1]), row[-1]]

        if len(row) == 7:
            rows.append(row)

df = pd.DataFrame(rows, columns=header)
df.columns = df.columns.str.strip()

# =========================
# FUNCIONES AUXILIARES
# =========================
def generar_timestamp(mes_str):
    meses = {
        "ene": 1,
        "feb": 2,
        "mar": 3,
        "abr": 4,
        "may": 5,
        "jun": 6,
        "jul": 7,
        "ago": 8,
        "sep": 9,
        "oct": 10,
        "nov": 11,
        "dic": 12
    }

    if pd.isna(mes_str):
        return None

    mes_str = str(mes_str).strip().lower()

    if "-" not in mes_str:
        return None

    partes = mes_str.split("-")

    if len(partes) != 2:
        return None

    mes_abbr, anio_short = partes

    if mes_abbr not in meses:
        return None

    try:
        anio = 2000 + int(anio_short)
    except:
        return None

    mes = meses[mes_abbr]

    inicio = datetime(anio, mes, 1)

    if mes == 12:
        fin = datetime(anio + 1, 1, 1)
    else:
        fin = datetime(anio, mes + 1, 1)

    delta = fin - inicio
    segundos_random = random.randint(0, int(delta.total_seconds()) - 1)

    return inicio + timedelta(seconds=segundos_random)

def generar_usuario():
    return f"USR{random.randint(1, NUM_USUARIOS_POOL):06d}"

def generar_ip():
    return str(ipaddress.IPv4Address(random.randint(0x0B000000, 0xDF000000)))

def generar_texto(tipo_queja):
    templates = TEXT_TEMPLATES.get(
        tipo_queja,
        ["Reclamo generado automáticamente."]
    )
    return random.choice(templates)



# =========================
# EXPANSIÓN
# =========================
registros = []

for _, row in df.iterrows():

    if pd.isna(row["N° de Quejas"]) or row["N° de Quejas"] == "":
        continue

    cantidad = int(float(row["N° de Quejas"]))

    for _ in range(cantidad):
        timestamp = generar_timestamp(row["Mes"])

        if timestamp is None:
            continue
        registros.append({
            "timestamp": timestamp,
            "usuario_id": generar_usuario(),
            "ip_address": generar_ip(),
            "empresa": row["Empresa operadora"],
            "departamento": row["Departamento"],
            "servicio": row["Servicio involucrado"],
            "medio_presentacion": row["Medio de presentación"],
            "tipo_queja": row["Tipo de Queja"],
            "texto_reclamo": generar_texto(row["Tipo de Queja"]),
            "is_synthetic_spam": 0
        })

# =========================
# DATAFRAME FINAL
# =========================
df_expandido = pd.DataFrame(registros)

df_expandido.insert(
    0,
    "id_reclamo",
    range(1, len(df_expandido) + 1)
)

df_expandido.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset expandido generado con {len(df_expandido):,} registros.")

Dataset expandido generado con 44,163 registros.


In [1]:
import pandas as pd
import random
import ipaddress
from datetime import timedelta

# =========================
# CONFIG
# =========================
INPUT_FILE = "dataset_expandido.csv"
OUTPUT_FILE = "dataset_1M_raw.csv"

TARGET_SIZE = 1_000_000

# Ruido / anomalías
NOISE_PERCENT = 0.10
SPAM_PERCENT = 0.05

INVALID_IP_PERCENT = 0.015
INVALID_TIMESTAMP_PERCENT = 0.015
EMPTY_FIELD_PERCENT = 0.01
DUPLICATE_PERCENT = 0.03

NUM_USUARIOS_POOL = 300000

random.seed(42)

# =========================
# CARGAR DATA BASE
# =========================
df = pd.read_csv(INPUT_FILE)

base_records = df.to_dict("records")

for r in base_records:
    r["timestamp"] = pd.to_datetime(r["timestamp"])

registros = base_records.copy()

# =========================
# HELPERS
# =========================
def generar_usuario():
    return f"USR{random.randint(1, NUM_USUARIOS_POOL):06d}"


def generar_ip():
    return str(
        ipaddress.IPv4Address(
            random.randint(0x0B000000, 0xDF000000)
        )
    )


def generar_ip_invalida():
    return random.choice([
        "999.999.999.999",
        "abc.def.ghi.jkl",
        "256.300.1.1",
        "",
        "IP_INVALIDA"
    ])


def generar_timestamp_invalido():
    return random.choice([
        "32/13/2016 99:99",
        "fecha_invalida",
        "",
        "2016-99-99 25:61:61"
    ])


def variar_timestamp(ts):
    delta = timedelta(
        days=random.randint(-5, 5),
        hours=random.randint(-12, 12),
        minutes=random.randint(-59, 59)
    )
    return ts + delta


def mutar_texto(texto):
    texto = str(texto)

    variantes = [
        texto,
        texto.lower(),
        texto.upper(),
        texto + " ",
        " " + texto,
        texto.replace("o", "0", 1),
        texto.replace("a", "@", 1),
        texto + "!!!"
    ]

    return random.choice(variantes)


def mutar_categoria(valor):
    valor = str(valor)

    variantes = [
        valor,
        valor.lower(),
        valor.title(),
        valor.strip(),
        valor + " "
    ]

    return random.choice(variantes)

# =========================
# GENERACIÓN
# =========================
next_id = len(registros) + 1

while len(registros) < TARGET_SIZE:

    base = random.choice(base_records)

    # -------------------------
    # DUPLICADOS INTENCIONALES
    # -------------------------
    if random.random() < DUPLICATE_PERCENT:
        duplicado = base.copy()
        duplicado["id_reclamo"] = next_id
        next_id += 1

        registros.append(duplicado)
        continue

    nuevo = base.copy()

    # -------------------------
    # VARIACIONES BASE
    # -------------------------
    nuevo["timestamp"] = variar_timestamp(base["timestamp"])
    nuevo["usuario_id"] = generar_usuario()
    nuevo["ip_address"] = generar_ip()

    # -------------------------
    # MUTACIONES LEVES
    # -------------------------
    if random.random() < 0.70:
        nuevo["texto_reclamo"] = mutar_texto(base["texto_reclamo"])

    if random.random() < 0.20:
        nuevo["departamento"] = mutar_categoria(base["departamento"])

    if random.random() < 0.20:
        nuevo["medio_presentacion"] = mutar_categoria(base["medio_presentacion"])

    # -------------------------
    # RUIDO SEMÁNTICO
    # -------------------------
    if random.random() < NOISE_PERCENT:

        campo_ruido = random.choice([
            "texto_reclamo",
            "departamento",
            "medio_presentacion"
        ])

        nuevo[campo_ruido] = mutar_texto(nuevo[campo_ruido])

    # -------------------------
    # SPAM SINTÉTICO
    # -------------------------
    if random.random() < SPAM_PERCENT:

        nuevo["is_synthetic_spam"] = 1

        nuevo["texto_reclamo"] = random.choice([
            "Necesito ayuda urgente",
            "Reclamo reclamo reclamo",
            "No responden",
            "Atención inmediata por favor",
            "URGENTE URGENTE"
        ])

        nuevo["timestamp"] = base["timestamp"]

    else:
        nuevo["is_synthetic_spam"] = 0

    # -------------------------
    # CORRUPCIÓN REAL
    # -------------------------
    if random.random() < INVALID_IP_PERCENT:
        nuevo["ip_address"] = generar_ip_invalida()

    if random.random() < INVALID_TIMESTAMP_PERCENT:
        nuevo["timestamp"] = generar_timestamp_invalido()

    if random.random() < EMPTY_FIELD_PERCENT:
        campo = random.choice([
            "usuario_id",
            "texto_reclamo",
            "empresa"
        ])
        nuevo[campo] = ""

    # -------------------------
    # ID FINAL
    # -------------------------
    nuevo["id_reclamo"] = next_id
    next_id += 1

    registros.append(nuevo)

    # Progreso visual
    if len(registros) % 50000 == 0:
        print(f"Generados: {len(registros):,}")

# =========================
# EXPORTAR
# =========================
df_final = pd.DataFrame(registros)

df_final["timestamp"] = df_final["timestamp"].astype(str)

df_final.to_csv(OUTPUT_FILE, index=False)

print(f"\nDataset final generado con {len(df_final):,} registros.")

Generados: 50,000
Generados: 100,000
Generados: 150,000
Generados: 200,000
Generados: 250,000
Generados: 300,000
Generados: 350,000
Generados: 400,000
Generados: 450,000
Generados: 500,000
Generados: 550,000
Generados: 600,000
Generados: 650,000
Generados: 700,000
Generados: 750,000
Generados: 800,000
Generados: 850,000
Generados: 900,000
Generados: 950,000
Generados: 1,000,000

Dataset final generado con 1,000,000 registros.
